### Montra Hospitality Intel Analysis


In [1]:
%pip install --upgrade sqlalchemy
%pip install python-dotenv
%pip install "psycopg[binary]"

  Using cached sqlalchemy-2.1.6-cp312-cp312-win_amd64.whl.metadata (9.9 kB)
Using cached sqlalchemy-2.1.6-cp312-cp312-win_amd64.whl (2.4 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached psycopg-3.3.6-py3-none-any.whl.metadata (4.4 kB)
  Using cached psycopg_binary-3.3.6-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
Using cached psycopg_binary-3.3.6-cp312-cp312-win_amd64.whl (3.7 MB)
Using cached psycopg-3.3.6-py3-none-any.whl (215 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importing libraries
import os 
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px
import html
import re
import seaborn as sns
from collections import Counter
import plotly
import re
import plotly.graph_objects as go
import plotly.subplots as sub
from plotly.subplots import make_subplots as msub 
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from turtle import title
from dotenv import load_dotenv

# ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# RAW, PROC, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "outputs"
# for p in (PROC, OUT): p.mkdir(parents=True, exist_ok=True)
SEED = 42
LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#D1495B", "neutral": "#B8BCC4", "positive": "#2E8B57"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# Load environment variables from .env
load_dotenv()

# Get the database URL
DATABASE_URL = os.getenv("DATABASE_URL")

# Check that the URL exists
if not DATABASE_URL:
    raise ValueError("DATABASE_URL is missing from the .env file")

# Create the database connection
engine = create_engine(DATABASE_URL,connect_args={"sslmode": "require", "connect_timeout": 10})

# Test the connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))

        print("Database response:", result.scalar())
        print("Successful connection!")

except Exception as err:
    print("Connection failed:", str(err))

Database response: 1
Successful connection!


In [4]:
# Querying the database
query = """ 
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""
# Load the attributes of database schema
table = pd.read_sql(query, engine)
table.head()


,table_name
0,bookings
1,guests
2,hotels
3,reviews
4,clean_bookings


#### Load Bookings data

In [5]:
# Accessing Hotel Bookings
booking_data = pd.read_sql("SELECT * FROM bookings", engine)
print("-"*30)
print(f"Hotel Bookings: {len(booking_data):,} rows, {booking_data.shape[1]} columns")

# Data Summary
bookings_summary = pd.DataFrame({
    "Data Type": booking_data.dtypes.astype(str),
    "Missing Values": booking_data.isna().sum(),
    "% Missing Values": (booking_data.isna().mean() * 100).round(2),
    "Unique Values": booking_data.nunique(),
    "Duplicate Values": [booking_data.duplicated(subset=[col]).sum() for col in booking_data.columns]
}).sort_values(by="% Missing Values", ascending=False)


# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/Raw_dataset")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
booking_data.to_csv(OUT_DATA_DIR / "booking_data.csv", index=False)
print("Bookings saved successfully!")
print("File:",(OUT_DATA_DIR / "booking_data.csv").resolve())

bookings_summary

------------------------------
Hotel Bookings: 35,000 rows, 9 columns
Bookings saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Dataset\Raw_dataset\booking_data.csv


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
booking_id,int64,0,0.0,35000,0
guest_id,int64,0,0.0,11377,23623
hotel_id,int64,0,0.0,25,34975
room_type,str,0,0.0,6,34994
booking_channel,str,0,0.0,7,34993
check_in_date,str,0,0.0,531,34469
check_out_date,str,0,0.0,531,34469
total_amount,float64,0,0.0,28357,6643
stay_duration,int64,0,0.0,6,34994


#### Load Guests Dataset

In [6]:
# Accessing Hotel Guests
guest_data = pd.read_sql("SELECT * FROM guests", engine)
print("-"*30)
print(f"Hotel Guests: {len(guest_data):,} rows, {guest_data.shape[1]} columns")

# Data Summary
guest_summary = pd.DataFrame({
    "Data Type": guest_data.dtypes.astype(str),
    "Missing Values": guest_data.isna().sum(),
    "% Missing Values": (guest_data.isna().mean() * 100).round(2),
    "Unique Values": guest_data.nunique(),
    "Duplicate Values": [guest_data.duplicated(subset=[col]).sum() for col in guest_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/Raw_dataset")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
guest_data.to_csv(OUT_DATA_DIR / "guest_data.csv", index=False)
print("Bookings saved successfully!")
print("File:",(OUT_DATA_DIR / "guest_data.csv").resolve())

guest_summary

------------------------------
Hotel Guests: 12,000 rows, 5 columns
Bookings saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Dataset\Raw_dataset\guest_data.csv


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
loyalty_status,str,6036,50.3,3,11996
guest_id,int64,0,0.0,12000,0
customer_type,str,0,0.0,5,11995
country,str,0,0.0,15,11985
previous_stays,int64,0,0.0,13,11987


#### Load Hotel Dataset

In [7]:
# Accessing Hotels
hotel_data = pd.read_sql("SELECT * FROM hotels", engine)
print("-"*30)
print(f"Total Hotels: {len(hotel_data):,} rows, {hotel_data.shape[1]} columns")

# Data Summary
hotel_summary = pd.DataFrame({
    "Data Type": hotel_data.dtypes.astype(str),
    "Missing Values": hotel_data.isna().sum(),
    "% Missing Values": (hotel_data.isna().mean() * 100).round(2),
    "Unique Values": hotel_data.nunique(),
    "Duplicate Values": [hotel_data.duplicated(subset=[col]).sum() for col in hotel_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/Raw_dataset")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
hotel_data.to_csv(OUT_DATA_DIR / "hotel_data.csv", index=False)
print("Bookings saved successfully!")
print("File:",(OUT_DATA_DIR / "hotel_data.csv").resolve())


hotel_summary

------------------------------
Total Hotels: 25 rows, 5 columns
Bookings saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Dataset\Raw_dataset\hotel_data.csv


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
hotel_id,int64,0,0.0,25,0
hotel_name,str,0,0.0,24,1
location,str,0,0.0,14,11
star_rating,int64,0,0.0,3,22
total_rooms,int64,0,0.0,23,2


In [8]:
# Accessing Hotel Reviews
review_data = pd.read_sql("SELECT * FROM reviews", engine)
print("-"*30)
print(f"Hotel Reviews: {len(review_data):,} rows, {review_data.shape[1]} columns")

# Data Summary
reviews_summary = pd.DataFrame({
    "Data Type": review_data.dtypes.astype(str),
    "Missing Values": review_data.isna().sum(),
    "% Missing Values": (review_data.isna().mean() * 100).round(2),
    "Unique Values": review_data.nunique(),
    "Duplicate Values": [review_data.duplicated(subset=[col]).sum() for col in review_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/Raw_dataset")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
review_data.to_csv(OUT_DATA_DIR / "review_data.csv", index=False)
print("Bookings saved successfully!")
print("File:",(OUT_DATA_DIR / "review_data.csv").resolve())


reviews_summary

------------------------------
Hotel Reviews: 35,000 rows, 8 columns
Bookings saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Dataset\Raw_dataset\review_data.csv


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
review_id,int64,0,0.0,35000,0
booking_id,int64,0,0.0,35000,0
guest_id,int64,0,0.0,11377,23623
hotel_id,int64,0,0.0,25,34975
review_text,str,0,0.0,34536,464
rating_score,float64,0,0.0,41,34959
review_date,str,0,0.0,545,34455
sentiment_label,str,0,0.0,3,34997


#### Data Summary:

In [9]:
# Load all dataset and merge them using refrence dataset (review_data)
# Load the datasets (adjust paths/format to your files)
review_data  = pd.read_csv(r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\Raw_dataset\review_data.csv")
booking_data = pd.read_csv(r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\Raw_dataset\booking_data.csv")
guest_data   = pd.read_csv(r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\Raw_dataset\guest_data.csv")
hotel_data   = pd.read_csv(r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\Raw_dataset\hotel_data.csv")

# Merge the four datasets
# validate="m:1" raises an error if the right-hand key is not unique,
# preventing silent row duplication

merged_data = (review_data
    .merge(booking_data, on="booking_id", how="left",
           validate="m:1", suffixes=("", "_booking_data"))
    .merge(guest_data, on="guest_id", how="left",
           validate="m:1", suffixes=("", "_guest_data"))
    .merge(hotel_data, on="hotel_id", how="left",
           validate="m:1", suffixes=("", "_hotel_data"))
)

# Check for duplicate columns while joining
assert len(merged_data) == len(review_data), "Merge changed the number of rows"

# Generate data summary
merged_summary = pd.DataFrame({
    "Data Type": merged_data.dtypes.astype(str),
    "Missing Values": merged_data.isna().sum(),
    "% Missing Values": (merged_data.isna().mean() * 100).round(2),
    "Unique Values": merged_data.nunique(),
    "Duplicate Values": merged_data.apply(lambda col: col.duplicated().sum()),
})

# Sort by missing percentage
merged_summary = merged_summary.sort_values(
    by="% Missing Values", ascending=False
)

# Display summary
display(merged_summary)

# Fully duplicated rows (usually the more useful duplicate check)
print(f"Fully duplicated rows: {merged_data.duplicated().sum()}")
print(f"Data Summary: {len(merged_data):,} rows, {merged_data.shape[1]} columns")

,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
loyalty_status,str,17615,50.33,3,34996
review_id,int64,0,0.00,35000,0
guest_id,int64,0,0.00,11377,23623
booking_id,int64,0,0.00,35000,0
review_text,str,0,0.00,34536,464
rating_score,float64,0,0.00,41,34959
review_date,str,0,0.00,545,34455
hotel_id,int64,0,0.00,25,34975
sentiment_label,str,0,0.00,3,34997
guest_id_booking_data,int64,0,0.00,11377,23623


Fully duplicated rows: 0
Data Summary: 35,000 rows, 24 columns


In [10]:
merged_data.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,guest_id_booking_data,hotel_id_booking_data,...,total_amount,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,6699,18,...,296.44,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,11380,23,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,4812,13,...,415.59,2,Business,NaN,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,982,12,...,601.18,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,615,5,...,603.85,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214


# Explatory Data Analysis

In [11]:
####### 1. ----------Booking Analysis Overview----------
#=================================================

# Total Booked Room_type
rm_type = merged_data['room_type'].value_counts().reset_index()
print(f"Total Rooms Booked:\n{rm_type}")
print("-"* 30)


###------ Revenue generated by room type
rm_revenue = merged_data.groupby('room_type')['total_amount'].sum().reset_index()
print(f" Total Rvenue Generated: \n{rm_revenue}")
print("-"* 35)

###------- Booking Channel Distribution
channels = merged_data['booking_channel'].value_counts().reset_index()
print(f"Total Booking Channel Counts: \n{channels}")
print("-"* 30)

###------- Revenue generated by booking channels
bc_revenue = merged_data.groupby('booking_channel')['total_amount'].sum().reset_index()
print(f"Book Channel Revenue: \n{bc_revenue}")
print("-"* 30)

###------ Check for guest stay duration
stay_duration = (
    merged_data["stay_duration"].value_counts().rename_axis("stay_duration")
    .reset_index(name="count").sort_values("stay_duration"))

print(f"Guest Stay Duration:\n {stay_duration}")


Total Rooms Booked:
     room_type  count
0     Standard  10596
1       Deluxe   8619
2    Executive   5293
3  Family Room   4137
4        Suite   3544
5         Twin   2811
------------------------------
 Total Rvenue Generated: 
     room_type  total_amount
0       Deluxe    3938732.66
1    Executive    3211430.94
2  Family Room    2158312.62
3     Standard    3112788.77
4        Suite    3029593.14
5         Twin     861141.58
-----------------------------------
Total Booking Channel Counts: 
    booking_channel  count
0    Direct Website   8688
1       Booking.com   7808
2           Expedia   5273
3      Travel Agent   3529
4            Airbnb   3416
5  Corporate Portal   3411
6           Walk-in   2875
------------------------------
Book Channel Revenue: 
    booking_channel  total_amount
0            Airbnb    1604609.36
1       Booking.com    3654237.93
2  Corporate Portal    1611396.00
3    Direct Website    4051385.20
4           Expedia    2452361.55
5      Travel Agent    16

In [12]:
####----------Visualization of Bookings
fig = msub(rows =3,cols= 2, 
           subplot_titles= ("Total Room Distribution", "Revenue Generated by Room Type", "Guest Booking Channels",
                            "Revenue Generated by Booking Channels", "Stay Duration Distribution (Days)", ),
           specs=[[{}, {}], [{}, {}], [{"colspan": 2}, None]])

## Add Charts
fig.add_trace(go.Bar(x= rm_type['room_type'], y = rm_type['count'], name = 'Counts', marker_color = 'teal'), row = 1, col = 1)
fig.add_trace(go.Bar(x= rm_revenue['room_type'], y = rm_revenue['total_amount'], name = 'Revenue', marker_color ='indianred'), row = 1, col = 2)
fig.add_trace(go.Bar(x= channels['booking_channel'], y = channels['count'], name = 'Counts', marker_color = 'blue'), row = 2, col = 1)
fig.add_trace(go.Bar(x= bc_revenue['booking_channel'], y = bc_revenue['total_amount'], name = 'Reveneue', marker_color = 'gold'), row =2, col =2)
fig.add_trace(go.Bar(x= stay_duration['stay_duration'], y =stay_duration['count'], name = 'Days Booked', marker_color = 'goldenrod'), row =3, col = 1)

fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  height = 800, title_text = 'Booking Analysis Overview', showlegend = False)


In [13]:
####### 2. ----------Guest Analysis Overview----------
#=====================================================

# Total Guest Class
cust_type = merged_data['customer_type'].value_counts().reset_index()
print(f"Total Guest Class:\n{cust_type}")
print("-"* 30)

# Most Patronised Guest Status
ly_status = merged_data['loyalty_status'].value_counts().reset_index()
print(f"Total Loyal Guest:\n{ly_status}")
print("-"* 30)


###------ Revenue generated by country
cn_revenue = merged_data.groupby('country')['total_amount'].sum().reset_index()
print(f" Total Revenue Generated: \n{cn_revenue}")
print("-"* 35)


###------- most loyal guest class
# cust_status = cleaned_data.groupby('customer_type')['loyalty_status'].value_counts().reset_index()
# print(f"Guest loyalty Status: \n{cust_status}")
# print("-"* 30)

cust_status = (
    merged_data
    .groupby(['customer_type', 'loyalty_status'])
    .size()
    .reset_index(name='count')
)


Total Guest Class:
  customer_type  count
0       Leisure  14000
1      Business   8404
2     Corporate   5517
3         Group   3564
4        Family   3515
------------------------------
Total Loyal Guest:
  loyalty_status  count
0         Silver   8605
1           Gold   5857
2       Platinum   2923
------------------------------
 Total Revenue Generated: 
                 country  total_amount
0              Australia      79593.78
1                 Canada     588749.02
2                  China      91383.47
3                 France     942353.12
4                Germany    1195641.84
5                  India     169334.61
6                Ireland     840061.73
7                  Italy     493253.12
8            Netherlands     783906.08
9                Nigeria    1604963.98
10          South Africa     340941.83
11                 Spain     889166.11
12  United Arab Emirates     631697.45
13        United Kingdom    5663730.72
14         United States    1997222.85
---------------

In [14]:
####----------Visualization of Guest
fig = msub(rows =2,cols= 2, 
           subplot_titles= ("Guest Class Distribution", "Guest Loyalty status", "Revenue Generated by Country",
                            "Guest loyalty Status"),
           specs=[[{}, {}], [{}, {}]])

## Add Charts
fig.add_trace(go.Bar(x= cust_type['customer_type'], y = cust_type['count'], name = 'Counts', marker_color = 'teal'), row = 1, col = 1)
fig.add_trace(go.Bar(x= ly_status['loyalty_status'], y = ly_status['count'], name = 'Counts', marker_color ='indianred'), row = 1, col = 2)
fig.add_trace(go.Bar(x= cn_revenue['country'], y = cn_revenue['total_amount'], name = 'Revenue', marker_color = 'blue'), row = 2, col = 1)
# Row 2, col 2: one Bar trace per loyalty status, stacked
colors = ['gold', 'blue', 'red', 'green', 'gray']
for color, (status, grp) in zip(colors, cust_status.groupby('loyalty_status')):
    fig.add_trace(go.Bar(x=grp['customer_type'], y=grp['count'], name=status, marker_color=color),
        row=2, col=2)

fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  height = 800, title_text = 'Guest Analysis Overview', barmode= 'stack', showlegend = False)


In [15]:
####### 3. ----------Hotel Analysis Overview----------
#=====================================================

# Total Booked Hotels
h_count = merged_data['hotel_name'].value_counts().reset_index()
print(f"Total Booked Hotels:\n{h_count}")
print("-"* 30)


###------ Revenue generated by by Hotels
h_revenue = merged_data.groupby('hotel_name')['total_amount'].sum().reset_index()
print(f" Total Revenue Generated: \n{h_revenue}")
print("-"* 35)


###------- Total Hotel Roooms
h_rooms = merged_data.groupby('hotel_name')['total_rooms'].value_counts().reset_index()
print(f"Book Channel Revenue: \n{h_rooms}")
print("-"* 30)

###------ Hotel Ratings
h_star_ratings = (
    merged_data.groupby("hotel_name")['star_rating'].value_counts().reset_index(name="count").sort_values("star_rating"))

print(f"Hotel Rating:\n {stay_duration}")


Total Booked Hotels:
                    hotel_name  count
0    Edinburgh Riverside Hotel   2909
1       Southampton Park Hotel   1460
2        Glasgow Skyline Hotel   1445
3   Nottingham Riverside Hotel   1442
4        Nottingham Park Hotel   1430
5         Glasgow Metropolitan   1428
6         The Belfast Heritage   1427
7          The London Boutique   1418
8      Manchester Metropolitan   1418
9         The Belfast Boutique   1409
10         The Oxford Boutique   1405
11     The Birmingham Heritage   1405
12       The Brighton Heritage   1399
13     Liverpool Harbour Hotel   1398
14        The Birmingham Grand   1390
15         The London Heritage   1389
16       Glasgow Harbour Hotel   1385
17             The Leeds Grand   1373
18        Belfast Metropolitan   1366
19          The Aberdeen Grand   1362
20        The Glasgow Heritage   1354
21     Leicester Harbour Hotel   1353
22         London Metropolitan   1338
23      Liverpool Metropolitan   1297
-----------------------------

In [16]:
####----- Visualzation of the Best Hotels
TOP_N = 10

# Top 10 for each chart
top_count   = h_count.nlargest(TOP_N, 'count')
top_revenue = h_revenue.nlargest(TOP_N, 'total_amount')
top_rooms   = h_rooms.nlargest(TOP_N, 'total_rooms')

# Top 10 hotels by total count, then keep only their star-rating rows
top_hotels = (
    h_star_ratings.groupby('hotel_name')['count'].sum().nlargest(TOP_N).index)
top_ratings = h_star_ratings[h_star_ratings['hotel_name'].isin(top_hotels)]

fig = msub(rows=2, cols=2,subplot_titles=("10 Most Booked Hotels", "Top 10 Hotels by Revenue",
                    "Top 10 Hotels by Rooms", "Best 10 Most Rated Hotels"),specs=[[{}, {}], [{}, {}]])

# Add charts
fig.add_trace(go.Bar(x=top_count['hotel_name'], y=top_count['count'],
                     name='Counts', marker_color='teal', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(x=top_revenue['hotel_name'], y=top_revenue['total_amount'],
                     name='Revenue', marker_color='indianred', showlegend=False), row=1, col=2)

fig.add_trace(go.Bar(x=top_rooms['hotel_name'], y=top_rooms['total_rooms'],
                     name='Hotel Rooms', marker_color='blue', showlegend=False), row=2, col=1)

# Top 10 Best Hotels
colors = ['gold', 'blue', 'red', 'green', 'gray']
for color, (star, grp) in zip(colors, top_ratings.groupby('star_rating')):
    fig.add_trace(go.Bar(x=grp['hotel_name'], y=grp['count'],
                         name=f"{star} star", marker_color=color), row=2, col=2)

fig.update_xaxes(tickangle=-45)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
    height=800, title_text='Hotel Analysis Overview', barmode='stack', showlegend=True,)

In [17]:
####### 4. ----------Hotel Review Analysis Overview----------
#============================================================

# Total Booked Room_type
rm_type = merged_data['room_type'].value_counts().reset_index()
print(f"Total Rooms Booked:\n{rm_type}")
print("-"* 30)


###------ Revenue generated by room type
rm_revenue = merged_data.groupby('room_type')['total_amount'].sum().reset_index()
print(f" Total Rvenue Generated: \n{rm_revenue}")
print("-"* 35)

###------- Booking Channel Distribution
channels = merged_data['booking_channel'].value_counts().reset_index()
print(f"Total Booking Channel Counts: \n{channels}")
print("-"* 30)

###------- Revenue generated by booking channels
s_rating = merged_data.groupby('sentiment_label')['rating_score'].value_counts().reset_index()
print(f"Hotel Rating by Sentiments: \n{s_rating}")
print("-"* 30)

###------ Check for guest stay duration
stay_duration = (
    merged_data["stay_duration"].value_counts().rename_axis("stay_duration")
    .reset_index(name="count").sort_values("stay_duration"))

print(f"Guest Stay Duration:\n {stay_duration}")


Total Rooms Booked:
     room_type  count
0     Standard  10596
1       Deluxe   8619
2    Executive   5293
3  Family Room   4137
4        Suite   3544
5         Twin   2811
------------------------------
 Total Rvenue Generated: 
     room_type  total_amount
0       Deluxe    3938732.66
1    Executive    3211430.94
2  Family Room    2158312.62
3     Standard    3112788.77
4        Suite    3029593.14
5         Twin     861141.58
-----------------------------------
Total Booking Channel Counts: 
    booking_channel  count
0    Direct Website   8688
1       Booking.com   7808
2           Expedia   5273
3      Travel Agent   3529
4            Airbnb   3416
5  Corporate Portal   3411
6           Walk-in   2875
------------------------------
Hotel Rating by Sentiments: 
   sentiment_label  rating_score  count
0         negative           2.4   1144
1         negative           2.3   1017
2         negative           2.2    969
3         negative           2.1    943
4         negative     

In [18]:
# Convert review dates
merged_data["review_date"] = pd.to_datetime(merged_data["review_date"], errors="coerce")

# Create dashboard
fig = msub(rows=2, cols=3,
    subplot_titles=["Rating by Sentiment", "Sentiment Distribution", "Monthly Reviews", 
                    "Reviews per Hotel", "Average Hotel Rating", "Review Length"])

# Rating distribution by sentiment labels
for label in LABELS:
    fig.add_trace(go.Histogram(x=merged_data.loc[merged_data["sentiment_label"] == label, "rating_score"],
            name=label, marker_color=COLORS[label], nbinsx=40, opacity=0.85 ), row=1, col=1)

# Sentiment Label distribution
counts = merged_data["sentiment_label"].value_counts().reindex(LABELS, fill_value=0)

fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color=[COLORS[label] for label in counts.index],
        text=counts.values, textposition="outside", showlegend=False), row=1, col=2)

#  Monthly reviews
monthly = (merged_data.dropna(subset=["review_date"]).set_index("review_date").resample("MS").size())

fig.add_trace(go.Scatter(x=monthly.index, y=monthly.values, mode="lines+markers", showlegend=False), row=1, col=3)

# Reviews per hotel
h_reviews = merged_data.groupby("hotel_name").size().sort_values()

fig.add_trace(go.Bar(x=h_reviews.values, y=h_reviews.index, orientation="h", showlegend=False), row=2, col=1)

# Average rating per hotel
ratings = merged_data.groupby("hotel_name")["rating_score"].mean().dropna().sort_values()

fig.add_trace(go.Bar(x=ratings.values, y=ratings.index, orientation="h", marker_color="#F58518",
           showlegend=False), row=2, col=2)

# Review length in words
review_length = merged_data["review_text"].fillna("").str.split().str.len()

fig.add_trace(go.Histogram(x=review_length, nbinsx=40, marker_color="#72B7B2", showlegend=False), row=2, col=3)

# Chart labels and layout
fig.update_xaxes(title_text="Rating Score", row=1, col=1)
fig.update_xaxes(title_text="Sentiment", row=1, col=2)
fig.update_xaxes(title_text="Month", tickangle=-45, row=1, col=3)
fig.update_xaxes(title_text="Number of Reviews", row=2, col=1)
fig.update_xaxes(title_text="Average Rating", row=2, col=2)
fig.update_xaxes(title_text="Number of Words", row=2, col=3)


fig.update_layout(
    title="Review Analysis Overview",
    height=1000,
    width=1400,
    bargap=0.3,
    barmode="overlay",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()

In [19]:
####--------- Word Review Analysis by Sentiment Label-----####
STOP = set("the a an and or but of to in on at for with was were is are it we i our my this that as be had have very".split())

def top_words(texts, n=12):
    words = " ".join(texts.dropna()).lower()
    words = words.replace(".", " ").replace(",", " ").split()

    return Counter(word for word in words
        if word not in STOP and len(word) > 2).most_common(n)

# Get frequent words by rating group
negative = top_words(merged_data.loc[merged_data["rating_score"].isin([1, 2]), "review_text"])
neutral = top_words(merged_data.loc[merged_data["rating_score"] == 3, "review_text"])
positive = top_words(merged_data.loc[merged_data["rating_score"].isin([4, 5]), "review_text"])

# Match sentiment labels to word groups
word_groups = {"negative": negative, "neutral": neutral, "positive": positive}

fig = msub(rows=1, cols=3,
    subplot_titles=["Negative Reviews (1–2 Stars)", "Neutral Reviews (3 Stars)", "Positive Reviews (4–5 Stars)"])

for col, label in enumerate(LABELS, start=1):
    data = word_groups[label]
    words = [word for word, count in data][::-1]
    counts = [count for word, count in data][::-1]

    fig.add_trace(go.Bar(x=counts, y=words, orientation="h", marker_color=COLORS[label],
            text=counts, textposition="outside", showlegend=False), row=1, col=col)

fig.update_layout(
    title="Most Frequent Words in Hotel Reviews by Sentiment",
    height=550,
    width=1400,
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(title_text="Word Frequency", rangemode="tozero")
fig.update_yaxes(title_text="Word", col=1)

fig.show()

In [20]:
# Reviews where rating and text may disagree (important for labelling quality!)
NEG = ["rude", "dirty", "filthy", "terrible", "awful", "broken", "poor"]
POS = ["excellent", "friendly", "spotless", "perfect", "fantastic", "delicious"]

def has_any(t, words): return any(w in t.lower() for w in words)

suspect_high = merged_data[(merged_data["rating_score"] >= 4) & merged_data["review_text"].apply(lambda t: has_any(t, NEG))]
suspect_low = merged_data[(merged_data["rating_score"] <= 2) & merged_data["review_text"].apply(lambda t: has_any(t, POS))]
print("High rating but negative words:", len(suspect_high))
print("Low rating but positive words:", len(suspect_low))
suspect_high[["rating_score", "review_text"]].head(3)

High rating but negative words: 0
Low rating but positive words: 105


,rating_score,review_text


In [21]:
merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   review_id              35000 non-null  int64         
 1   booking_id             35000 non-null  int64         
 2   guest_id               35000 non-null  int64         
 3   hotel_id               35000 non-null  int64         
 4   review_text            35000 non-null  str           
 5   rating_score           35000 non-null  float64       
 6   review_date            35000 non-null  datetime64[us]
 7   sentiment_label        35000 non-null  str           
 8   guest_id_booking_data  35000 non-null  int64         
 9   hotel_id_booking_data  35000 non-null  int64         
 10  room_type              35000 non-null  str           
 11  booking_channel        35000 non-null  str           
 12  check_in_date          35000 non-null  str           
 13  check_out_da

#### Save Merged Dataset

In [ ]:
# Merged Data Summary
montra_summary = pd.DataFrame({
    "Data Type": merged_data.dtypes.astype(str),
    "Missing Values": merged_data.isna().sum(),
    "% Missing Values": (merged_data.isna().mean() * 100).round(2),
    "Unique Values": merged_data.nunique(),
    "Duplicate Values": [merged_data.duplicated(subset=[col]).sum() for col in merged_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

# OUTPUT DIRECTORIES
OUT_DATA_DIR = Path("../Dataset/raw_dataset")
OUT_ARTIFACTS_DIR = Path("../Montra_Hospitality_Intel_PJ/artifacts")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# SAVE Data
merged_data.to_csv(OUT_DATA_DIR / "merged_montra_data.csv", index=False)
print("Merged Montra Data saved successfully!")
print("File:",(OUT_DATA_DIR / "Merged_montra_data.csv").resolve())


montra_summary

Merged Montra Data saved successfully!
File: C:\Users\User\Documents\AMDARI_IMS\MONTRA_HOSPITALITY_PJ\Dataset\Raw_dataset\merged_montra_data.csv


,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
loyalty_status,str,17615,50.33,3,34996
review_id,int64,0,0.00,35000,0
guest_id,int64,0,0.00,11377,23623
booking_id,int64,0,0.00,35000,0
review_text,str,0,0.00,34536,464
rating_score,float64,0,0.00,41,34959
review_date,datetime64[us],0,0.00,545,34455
hotel_id,int64,0,0.00,25,34975
sentiment_label,str,0,0.00,3,34997
guest_id_booking_data,int64,0,0.00,11377,23623


In [23]:
merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   review_id              35000 non-null  int64         
 1   booking_id             35000 non-null  int64         
 2   guest_id               35000 non-null  int64         
 3   hotel_id               35000 non-null  int64         
 4   review_text            35000 non-null  str           
 5   rating_score           35000 non-null  float64       
 6   review_date            35000 non-null  datetime64[us]
 7   sentiment_label        35000 non-null  str           
 8   guest_id_booking_data  35000 non-null  int64         
 9   hotel_id_booking_data  35000 non-null  int64         
 10  room_type              35000 non-null  str           
 11  booking_channel        35000 non-null  str           
 12  check_in_date          35000 non-null  str           
 13  check_out_da